In [ ]:
import pandas as pd
import numpy as np
import csv
import matplotlib.pyplot as plt


def build_area_hour_category_feature_file(
    file_path,
    output_path=None,
    time_freq="h",
    use_full_day_grid=True,
    top_n_categories=None
):
    """
    Build a processed area-hour feature table for crime category prediction.

    Target:
    - crime_category = most frequent Primary Type in that area-hour
    - NO_CRIME if no incident occurred in that area-hour
    """

    # 1. Load only required columns safely
    try:
        df = pd.read_csv(
            file_path,
            usecols=["Date", "Community Area", "Primary Type"],
            engine="python",
            on_bad_lines="skip",
            encoding_errors="replace"
        )
    except Exception as e:
        print("Normal CSV read failed. Trying fallback reader...")
        print(e)

        df = pd.read_csv(
            file_path,
            usecols=["Date", "Community Area", "Primary Type"],
            engine="python",
            quoting=csv.QUOTE_NONE,
            on_bad_lines="skip",
            encoding_errors="replace"
        )

    # 2. Parse and clean
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Community Area"] = pd.to_numeric(df["Community Area"], errors="coerce")
    df["Primary Type"] = df["Primary Type"].astype(str).str.strip()

    df = df.dropna(subset=["Date", "Community Area", "Primary Type"]).copy()
    df["Community Area"] = df["Community Area"].astype(int)

    # Optional: keep only top N categories and merge others into OTHER
    if top_n_categories is not None:
        top_types = df["Primary Type"].value_counts().nlargest(top_n_categories).index
        df["Primary Type"] = df["Primary Type"].where(
            df["Primary Type"].isin(top_types),
            "OTHER"
        )

    # 3. Create hourly time slot
    df["hour_slot"] = df["Date"].dt.floor(time_freq)

    # 4. Aggregate crime count per area-hour
    crime_counts = (
        df.groupby(["Community Area", "hour_slot"])
          .size()
          .reset_index(name="crime_count")
    )

    crime_counts["crime_occurrence"] = (
        crime_counts["crime_count"] > 0
    ).astype(int)

    # 5. Find dominant crime type in each area-hour
    type_counts = (
        df.groupby(["Community Area", "hour_slot", "Primary Type"])
          .size()
          .reset_index(name="type_count")
    )

    dominant_type = (
        type_counts.sort_values(
            ["Community Area", "hour_slot", "type_count", "Primary Type"],
            ascending=[True, True, False, True]
        )
        .drop_duplicates(subset=["Community Area", "hour_slot"], keep="first")
        [["Community Area", "hour_slot", "Primary Type"]]
        .rename(columns={"Primary Type": "crime_category"})
    )

    # 6. Create full area-hour grid
    all_areas = sorted(df["Community Area"].unique())

    if use_full_day_grid:
        start_day = df["Date"].dt.floor("D").min()
        end_day = df["Date"].dt.floor("D").max()

        all_hours = pd.date_range(
            start=start_day,
            end=end_day + pd.Timedelta(hours=23),
            freq=time_freq
        )
    else:
        all_hours = pd.date_range(
            start=df["hour_slot"].min(),
            end=df["hour_slot"].max(),
            freq=time_freq
        )

    full_grid = pd.MultiIndex.from_product(
        [all_areas, all_hours],
        names=["Community Area", "hour_slot"]
    ).to_frame(index=False)

    # 7. Merge counts and dominant category
    data = full_grid.merge(
        crime_counts,
        on=["Community Area", "hour_slot"],
        how="left"
    )

    data = data.merge(
        dominant_type,
        on=["Community Area", "hour_slot"],
        how="left"
    )

    # Fill zero-crime rows
    data["crime_count"] = data["crime_count"].fillna(0)
    data["crime_occurrence"] = data["crime_occurrence"].fillna(0).astype(int)
    data["crime_category"] = data["crime_category"].fillna("NO_CRIME")

    # 8. Time features
    data["hour"] = data["hour_slot"].dt.hour
    data["day_of_week"] = data["hour_slot"].dt.dayofweek
    data["month"] = data["hour_slot"].dt.month
    data["day"] = data["hour_slot"].dt.day
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

    # 9. Cyclical hour encoding
    data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
    data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

    # 10. Sort before lag/rolling features
    data = data.sort_values(
        ["Community Area", "hour_slot"]
    ).reset_index(drop=True)

    # 11. Lag features
    grouped = data.groupby("Community Area")["crime_count"]

    data["lag_1h"] = grouped.shift(1)
    data["lag_2h"] = grouped.shift(2)
    data["lag_3h"] = grouped.shift(3)
    data["lag_24h"] = grouped.shift(24)

    # 12. Rolling features using past values only
    data["rolling_3h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(3).mean()
    )

    data["rolling_6h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(6).mean()
    )

    data["rolling_12h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(12).mean()
    )

    data["rolling_24h_mean"] = grouped.transform(
        lambda x: x.shift(1).rolling(24).mean()
    )

    # 13. Fill missing lag/rolling values
    lag_roll_cols = [
        "lag_1h", "lag_2h", "lag_3h", "lag_24h",
        "rolling_3h_mean", "rolling_6h_mean",
        "rolling_12h_mean", "rolling_24h_mean"
    ]

    data[lag_roll_cols] = data[lag_roll_cols].fillna(0)

    # 14. Type cleanup
    int_cols = [
        "Community Area",
        "crime_occurrence",
        "hour",
        "day_of_week",
        "month",
        "day",
        "is_weekend"
    ]

    for col in int_cols:
        data[col] = data[col].astype(int)

    # 15. Save if requested
    if output_path is not None:
        data.to_csv(output_path, index=False)

    return data

In [ ]:
processed_category_df = build_area_hour_category_feature_file(
    file_path="/content/Chicago_Crimes_SEIS764.csv")

print(processed_category_df.shape)
print(processed_category_df.columns)
print(processed_category_df["crime_category"].value_counts())
processed_category_df.head()

In [ ]:
target_col = "crime_category"

# feature columns
feature_cols = [
    "Community Area",
    "day_of_week",
    "month",
    "day",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "lag_1h",
    "lag_2h",
    "lag_3h",
    "lag_24h",
    "rolling_3h_mean",
    "rolling_6h_mean",
    "rolling_12h_mean",
    "rolling_24h_mean",
    "crime_occurrence"
]

# keep only required columns
model_df = processed_category_df[feature_cols + [target_col]].copy()


In [ ]:
model_df.head()

Section 1: Filter Actual Crime Category Records Only

In [ ]:
category_eda_df = processed_category_df.copy()

# Keep only rows where crime happened
category_eda_df = category_eda_df[
    category_eda_df["crime_occurrence"] == 1
].copy()

# Remove NO_CRIME category
category_eda_df = category_eda_df[
    category_eda_df["crime_category"] != "NO_CRIME"
].copy()

total_category_records = len(category_eda_df)

print("Total actual crime category records:", total_category_records)
print("\nCrime occurrence values after filtering:")
print(category_eda_df["crime_occurrence"].value_counts())

print("\nTop crime categories:")
print(category_eda_df["crime_category"].value_counts().head(10))

category_eda_df.head()

Section 2: Crime Category Distribution

In [ ]:
# ============================================================
# Section 2: Crime Category Distribution Among Actual Crimes
# ============================================================

category_counts = category_eda_df["crime_category"].value_counts()

category_summary = pd.DataFrame({
    "Crime_Category": category_counts.index,
    "Crime_Count": category_counts.values,
    "Percentage_of_Crime_Records": (
        category_counts.values / total_category_records
    ) * 100
})

display(category_summary)

# Donut chart: top 6 categories + OTHER
top6 = category_summary.head(6).copy()
other_count = category_summary.iloc[6:]["Crime_Count"].sum()
other_percentage = category_summary.iloc[6:]["Percentage_of_Crime_Records"].sum()

donut_df = pd.concat([
    top6,
    pd.DataFrame({
        "Crime_Category": ["OTHER"],
        "Crime_Count": [other_count],
        "Percentage_of_Crime_Records": [other_percentage]
    })
])

plt.figure(figsize=(7,7))

plt.pie(
    donut_df["Percentage_of_Crime_Records"],
    labels=donut_df["Crime_Category"],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops=dict(width=0.4)
)

plt.title("Crime Category Share Among Actual Crime Records")
plt.show()

Section 3: Top Crime Categories



In [ ]:
top10_categories = category_summary.head(10)

plt.figure(figsize=(9,6))

plt.hlines(
    y=top10_categories["Crime_Category"],
    xmin=0,
    xmax=top10_categories["Percentage_of_Crime_Records"]
)

plt.plot(
    top10_categories["Percentage_of_Crime_Records"],
    top10_categories["Crime_Category"],
    "o"
)

plt.title("Top Crime Categories Among Actual Crime Records")
plt.xlabel("Percentage of Crime Records (%)")
plt.ylabel("Crime Category")
plt.grid(True)
plt.show()

display(top10_categories)

Section 4: Top Crime Categories by Hour

In [ ]:
top5_categories = category_eda_df["crime_category"].value_counts().head(5).index

hour_category_counts = (
    category_eda_df[category_eda_df["crime_category"].isin(top5_categories)]
    .groupby(["hour", "crime_category"])
    .size()
    .unstack(fill_value=0)
)

# Convert to percentage within each hour
hour_category_percent = hour_category_counts.div(
    hour_category_counts.sum(axis=1),
    axis=0
) * 100

plt.figure(figsize=(11,6))

for category in hour_category_percent.columns:
    plt.plot(
        hour_category_percent.index,
        hour_category_percent[category],
        marker="o",
        label=category
    )

plt.title("Top Crime Categories by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Percentage Within Hour (%)")
plt.xticks(range(0,24))
plt.legend()
plt.grid(True)
plt.show()

Section 5: Crime Category by Hour

In [ ]:
top8_categories = category_eda_df["crime_category"].value_counts().head(8).index

category_hour_pivot = (
    category_eda_df[category_eda_df["crime_category"].isin(top8_categories)]
    .pivot_table(
        values="Community Area",
        index="crime_category",
        columns="hour",
        aggfunc="count",
        fill_value=0
    )
)

# Each category row sums to 100%
category_hour_percent = category_hour_pivot.div(
    category_hour_pivot.sum(axis=1),
    axis=0
) * 100

plt.figure(figsize=(14,6))

plt.imshow(category_hour_percent, aspect="auto")
plt.colorbar(label="Percentage Within Crime Category (%)")

plt.title("Crime Category Distribution by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Crime Category")

plt.xticks(range(24), range(24))
plt.yticks(
    range(len(category_hour_percent.index)),
    category_hour_percent.index
)

plt.show()

Section 6: Crime Category by Day of Week

In [ ]:
day_labels = {
    0: "Mon",
    1: "Tue",
    2: "Wed",
    3: "Thu",
    4: "Fri",
    5: "Sat",
    6: "Sun"
}

day_category_pivot = (
    category_eda_df[category_eda_df["crime_category"].isin(top8_categories)]
    .pivot_table(
        values="Community Area",
        index="crime_category",
        columns="day_of_week",
        aggfunc="count",
        fill_value=0
    )
)

day_category_percent = day_category_pivot.div(
    day_category_pivot.sum(axis=1),
    axis=0
) * 100

plt.figure(figsize=(10,6))

plt.imshow(day_category_percent, aspect="auto")
plt.colorbar(label="Percentage Within Crime Category (%)")

plt.title("Crime Category Distribution by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Crime Category")

plt.xticks(range(7), [day_labels[i] for i in range(7)])
plt.yticks(
    range(len(day_category_percent.index)),
    day_category_percent.index
)

plt.show()

Section 7: Crime Category by Community Area

In [ ]:
# ============================================================
# Section 7: Crime Category by Community Area
# ============================================================

# Use top 5 categories and top 10 community areas for readability
top5_categories = category_eda_df["crime_category"].value_counts().head(5).index
top10_areas = category_eda_df["Community Area"].value_counts().head(10).index

bubble_df = (
    category_eda_df[
        category_eda_df["crime_category"].isin(top5_categories) &
        category_eda_df["Community Area"].isin(top10_areas)
    ]
    .groupby(["Community Area", "crime_category"])
    .size()
    .reset_index(name="Crime_Count")
)

# Convert category and area to numeric positions
category_order = list(top5_categories)
area_order = list(top10_areas)

bubble_df["category_pos"] = bubble_df["crime_category"].apply(
    lambda x: category_order.index(x)
)

bubble_df["area_pos"] = bubble_df["Community Area"].apply(
    lambda x: area_order.index(x)
)

plt.figure(figsize=(11,6))

plt.scatter(
    bubble_df["category_pos"],
    bubble_df["area_pos"],
    s=bubble_df["Crime_Count"] * 3,
    alpha=0.6
)

plt.title("Crime Category Concentration by Community Area")
plt.xlabel("Crime Category")
plt.ylabel("Community Area")

plt.xticks(
    range(len(category_order)),
    category_order,
    rotation=30
)

plt.yticks(
    range(len(area_order)),
    area_order
)

plt.grid(True)
plt.show()

display(bubble_df.sort_values("Crime_Count", ascending=False).head(15))

Section 8: Most Common Crime Category by Community Area

In [ ]:


area_category_counts = (
    category_eda_df
    .groupby(["Community Area", "crime_category"])
    .size()
    .reset_index(name="Crime_Count")
)

dominant_category_by_area = (
    area_category_counts
    .sort_values(["Community Area", "Crime_Count"], ascending=[True, False])
    .drop_duplicates(subset=["Community Area"], keep="first")
)

print("Most Common Crime Category by Community Area")
display(dominant_category_by_area.head(20))

Section 9: Peak Hour and Peak Day by Crime Category

In [ ]:

# Peak hour by category
category_hour_counts = (
    category_eda_df
    .groupby(["crime_category", "hour"])
    .size()
    .reset_index(name="Crime_Count")
)

peak_hour_by_category = (
    category_hour_counts
    .sort_values(["crime_category", "Crime_Count"], ascending=[True, False])
    .drop_duplicates(subset=["crime_category"], keep="first")
)

# Peak day by category
category_day_counts = (
    category_eda_df
    .groupby(["crime_category", "day_of_week"])
    .size()
    .reset_index(name="Crime_Count")
)

peak_day_by_category = (
    category_day_counts
    .sort_values(["crime_category", "Crime_Count"], ascending=[True, False])
    .drop_duplicates(subset=["crime_category"], keep="first")
)

peak_day_by_category["Day_Name"] = peak_day_by_category["day_of_week"].map({
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
})

print("Peak Hour by Crime Category")
display(peak_hour_by_category.sort_values("Crime_Count", ascending=False).head(15))

print("Peak Day by Crime Category")
display(peak_day_by_category.sort_values("Crime_Count", ascending=False).head(15))

This category EDA uses only actual crime records. NO_CRIME and crime_occurrence = 0 records were removed so that the analysis focuses only on real crime categories.

The category distribution shows which crime types occur most often. The donut chart gives a quick summary of the largest crime categories, while the lollipop chart shows the top categories more clearly.

The hour-based analysis shows how common crime categories change across the day. The line chart compares the top categories by hour, while the heatmap shows which hours are more common within each crime category.

The day-of-week heatmap shows whether crime categories follow weekly patterns. The Community Area bubble chart shows which crime categories are concentrated in which areas. The tables identify the most common crime type in each Community Area and the peak hour or day for each crime category.

Overall, the EDA shows that crime categories are not evenly distributed. Some categories are much more common than others, and crime patterns vary by hour, day of week, and Community Area.